# SD1.5 Prompt-Mismatched In-Range Weighted Recovery-CFG Ablation

This two-trial study fixes each Christoffel sampling distribution at its CFG-7.5 S10000 estimate and varies only the recovery conditioning. All runs use the weighted unitary Fourier operator, $\zeta=1/2$, and the 2,000-step main weighted learning-rate schedule.


In [ ]:
from pathlib import Path
import importlib.util
import os

cwd = Path.cwd().resolve()
candidates = [cwd / 'analyze_results/weighted/ablation']
candidates.extend(parent / 'analyze_results/weighted/ablation' for parent in (cwd, *cwd.parents))
STUDY_ROOT = next((path for path in candidates if (path / 'analysis.py').is_file()), None)
if STUDY_ROOT is None:
    raise FileNotFoundError('Could not locate analyze_results/weighted/ablation/analysis.py')
spec = importlib.util.spec_from_file_location('weighted_cfg_ablation_analysis', STUDY_ROOT / 'analysis.py')
diagnostic = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(diagnostic)

SCENARIO = 'prompt_mismatched'
OUTPUT_DIR = diagnostic.RESULT_ROOT / SCENARIO / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_DIR / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
diagnostic.PROJECT_ROOT, STUDY_ROOT, OUTPUT_DIR


## Load Results

LPIPS is stored during new reconstructions and is also filled incrementally for any compatible legacy artifact. The completion table contains every expected sampling-law, recovery-CFG, ratio, and trial cell.


In [ ]:
LPIPS_TABLE = diagnostic.ensure_lpips(SCENARIO, device='cpu')
ROWS = diagnostic.load_rows(SCENARIO)
COMPLETION = diagnostic.completion_table(ROWS)
print(f'Loaded {len(ROWS)} / {diagnostic.EXPECTED_ROWS_PER_SCENARIO} reconstructions')
display(diagnostic.count_table(ROWS))
display(COMPLETION.groupby(['sampling_law', 'recovery_line'], as_index=False)[['observed', 'expected', 'left']].sum())
display(ROWS[['distribution_key', 'line_condition', 'samp_perc', 'repeat_id', 'psnr_db', 'ssim', 'lpips', 'pixel_mae']].head())


## Metric Curves

The TeX-styled figure matches the established ablation layout. Solid lines show two-trial arithmetic means and shading shows one sample standard deviation; with only two trials, this is more transparent than presenting a nominal confidence interval based on one degree of freedom.


In [ ]:
METRIC_OUTPUTS = diagnostic.plot_metric_curves(ROWS, output_dir=OUTPUT_DIR, show=True)
METRIC_OUTPUTS


## Reconstruction Panel

Each row is one fixed CFG-7.5 sampling law. Within each recovery column, the displayed trial minimizes LPIPS, with PSNR used to break ties. Change `PANEL_RATIO` to any value in the 1--5% grid.


In [ ]:
PANEL_RATIO = 0.01
PANEL_OUTPUTS = diagnostic.plot_reconstruction_panel(
    ROWS,
    sampling_ratio=PANEL_RATIO,
    output_dir=OUTPUT_DIR,
    show=True,
)
PANEL_OUTPUTS
